## **GE5219: CROWD DENSITY LIVE VECTOR MAP**


### STEP 1: DATA COLLECTION   

##### *1A) IMPORT LIBRARIES IN*

In [ ]:
import os #for creation of directories 
import requests #to fetch data from api (http)
import pandas as pd #for creation and manipulation of data (i.e dataframes)
from datetime import datetime #for timestamps (file naming & directory creation)

from arcgis.gis import GIS #to log into arcgis online 
from arcgis.features import FeatureLayerCollection #update hosted layers/tables
from arcgis.features import FeatureLayer

import glob #listing and archival of files

import json #common for conversion between Python dictionaries and JSON strings 
import boto3 #for cloud based service connections
import time #for time related function (adding delays, timestamps, runtime)
from botocore.exceptions import NoCredentialsError #to catch missing AWS credentials

##### *1B) LIVE DATA RETRIEVAL PREPARATION (DEFINING)*

In [ ]:
API_KEY = "wmJi+YXmRQacvwVXPAXKJA==" #fetching API data using unique ID
if not API_KEY:
    raise ValueError("Missing LTA_KEY") #incorporated condition that raises error if missing 

url = "https://datamall2.mytransport.sg/ltaodataservice/PCDRealTime" #URL provided on LTA Data Mall for Passenger Crowd Density (PCD) data
headers = {"AccountKey": API_KEY.strip(), "accept": "application/json"} #define request header where the key is needed for authorisation
train_lines = ["CCL","CEL","CGL","DTL","EWL","NEL","NSL","BPL","SLRT","PLRT","TEL"] #all mrt lines are listed to run query

#####  *1C) LIVE DATA RETRIEVAL PER MRT LINE THROUGH LOOPING*

In [ ]:
rows = [] #initialise list to store all records
print(f"\n Fetching live crowd density data from LTA ({datetime.now():%Y-%m-%d %H:%M:%S}) ...") #print timestamp message showing progress 

#With guidance from LTA's API Documentation...
#Using for loop to go through each station line by their code index
#A get request is sent to LTA API through the parameter "TrainLine"
#If successful: JSON response would be extracted as a station list. Each station is then recorded as a dictionary (station name, crowd level, train line, UTC timestamp)
#If not successful: prints failed error statement

for line in train_lines:
    r = requests.get(url, headers=headers, params={"TrainLine": line})
    if r.status_code == 200:
        vals = r.json().get("value", [])
        print(f"{line}: {len(vals)} records fetched")
        for v in vals:
            rows.append({
                "Station": v.get("Station",""),
                "CrowdLevel": v.get("CrowdLevel",""),
                "TrainLine": line,
                "Last_Updated": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            })
    else:
        print(f"  {line} failed ({r.status_code})")

# In the scenario that nothing is received, this code would be triggered
if not rows:
    raise RuntimeError("No data fetched")

### STEP 2: DATA SAVING AND ARCHIVING 

#####  *2A) DATAFRAME CREATION AND SAVING OF CSV*

In [ ]:
df = pd.DataFrame(rows) #convert rows to dataframe
csv_path = "LTA_station_crowd_density.csv" #define specific csv file 
df.to_csv(csv_path, index=False) #saves pandas dataframe as a csv file 
print(f"Saved: {csv_path}") #statement confirmation of saved file 

In [ ]:
archive_dir = r"C:\Users\Cheryl\OneDrive - National University of Singapore\NUS_CN\GE5219-SPATIAL PROGRAMMING\03_Projects\FINAL\Crowd Density Archive" #define archive folder path
timestamp = datetime.now().strftime("%Y%m%d_%H%M") #create timestamped filename
archive_path = os.path.join(archive_dir, f"LTA_station_crowd_density_{timestamp}.csv") #combining folder and name into one path
df.to_csv(archive_path, index=False) #save current dataframe to csv file
print(f" Archived > {archive_path}") #statement confirmation of saved file 

In [ ]:
max_files = 50 #setting limit on archive file numbers
files = sorted(glob.glob(os.path.join(archive_dir, "LTA_station_crowd_density_*.csv"))) 
if len(files) > max_files: #if files exceeded the max, select these files for deletion 
    for old_file in files[:-max_files]:
        os.remove(old_file) #deletes each file from archive
        print(f" Deleted archive: {old_file}") #statement confirmation of archive deletion

#####  *2B) CONNECT WITH ARCGIS ONLINE ENVIRONMENT*

In [ ]:
#defining credentials and item id where hosted table is 
ARCGIS_USER = "<username input-redacted due to privacy>"
ARCGIS_PASS = "<password input-redacted due to privacy>"
ITEM_ID = "5483652ac3b84c1b9643f9783e4cb20b"  

#authenticates session with defined credentials
gis = GIS("https://www.arcgis.com", ARCGIS_USER, ARCGIS_PASS)
#confirmation statement of successful login 
print(f" Logged in as: {gis.users.me.username}") 

#obtain target table from ArcGIS Online 
item = gis.content.get(ITEM_ID)
#exception written for statement to print if target table could not be retrieved
if not item:
    print(" Item_ID not found ")
#if found, confirmation statement printed
print(f"Found item: {item.title} ({item.type})")
flc = FeatureLayerCollection.fromitem(item)

#identifying target table/layer inside the collection for modification 
layer = None #initialise variable with a 'None' value
# if table exists, the table would be assigned to layer
if flc.tables:
    layer = flc.tables[0]
    print(f"Targeting Table: {layer.properties.name}")
# if table does not exist, the error statement would be printed
else:
    raise RuntimeError("No tables in this item")

#####  *2C) REPLACING OLD RECORDS WITH NEW UPDATES*

In [ ]:
print(" Clearing old rows ...") #statement confirmation that section has started running
try:
    layer.manager.truncate() #removing all rows from table 
    print("Table truncated.") #statement confirmation that existing records have been successfully cleared
    
# if truncation fails, all rows are deleted manually through a SQL true condition
except Exception as e:
    print(f"Truncate not allowed, deleting all rows: {e}")
    layer.delete_features(where="1=1")
    print(" All rows deleted.")

# convert DataFrame to ArcGIS Feature format
# iterating across DataFrame, converting each row into dictionary. each entry has a attribute key and field value
records = [
    {"attributes": {
        "Station": row["Station"],
        "CrowdLevel": row["CrowdLevel"],
        "TrainLine": row["TrainLine"],
        "Last_Updated": row["Last_Updated"]
    }}
    for _, row in df.iterrows()
]

# helper to split into 1000-feature batches
#1000 is the typical maximum limit of data one is able to send in to web APIs like ArcGIS Online
def chunk(it, n):
    for i in range(0, len(it), n): #loop through list in n steps
        yield it[i:i+n] #obtain n items per batch

# uploads each batch of features to ArcGIS layer 
adds_total = 0 #track counter for record addition 
for batch in chunk(records, 1000): #for loop goes through each group one by one 
    resp = layer.edit_features(adds=batch) #upload batch to layer per request

# summary log 
print(f"Successfully added {adds_total} rows to {layer.properties.name}.") #successful addition of rows
print(f"Completed run at {datetime.now():%Y-%m-%d %H:%M:%S}") #successful timestamp completion

### STEP 03: ARCPY PROCESSING WITH LIVE DATA

#####  3A) LINKING TABLE FROM ARCGIS ONLINE

In [ ]:
# Connect to active signed-in ArcGIS Pro session with credentials defined previously
gis = GIS('pro')

# Connect to hosted feature table URL containing crowd data
table_url = "https://services5.arcgis.com/KiRa9d9aHfdXiCqt/arcgis/rest/services/Stations_CD_Live/FeatureServer/14"
table = FeatureLayer(table_url)

# Query all rows within the hosted table (condition 1=1 means to return all)
# Query result is converted into spatially enabled data frame which is sdf
sdf = table.query(where="1=1").sdf

# Print record numbers once successful retrieval is confirmed
print(f"Connected successfully! {len(sdf)} records found.")

#####  3B) ENVIRONMENT SET UP & DEFINING WORKSPACE

In [ ]:
print(f"\n Started {datetime.datetime.now():%Y-%m-%d %H:%M:%S}") # represents when this section runs using timestamp

arcpy.env.overwriteOutput = True # allows automatic replacement of existing outputs 
arcpy.CheckOutExtension("Spatial")   # activates spatial analyst extention (this is needed for analysis tools)
arcpy.ClearWorkspaceCache_management() # clears any locks or cached connection in gdb
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(3414)  # sets working projection to Singapore's SVY21 (EPSG:3414)

# Defining Locations of Hosted Feature Layers
crowd_table    = "https://services5.arcgis.com/KiRa9d9aHfdXiCqt/arcgis/rest/services/Stations_CD_Live/FeatureServer/14"
station_points = "https://services5.arcgis.com/KiRa9d9aHfdXiCqt/arcgis/rest/services/Station_Point_Master/FeatureServer/0"
sg_boundary    = "https://services5.arcgis.com/KiRa9d9aHfdXiCqt/arcgis/rest/services/Singapore_Boundary_All/FeatureServer/0"

# Defining Locations of Local Workspace 
gdb_path = r"C:\Users\Cheryl\OneDrive - National University of Singapore\NUS_CN\GE5219-SPATIAL PROGRAMMING\03_Projects\FINAL\Crowd Density Raster\Crowd Density Raster.gdb"
os.makedirs(os.path.dirname(gdb_path), exist_ok=True) #ensure that output folder exists before saving files within
arcpy.env.workspace = gdb_path

# Defining Output Locations for the various files
join_numeric = os.path.join(gdb_path, "CDR_Join_Numeric")
buffer_fc    = os.path.join(gdb_path, "CDR_PointBuffers")
sg_local     = os.path.join(gdb_path, "SG_Boundary_Local")
output_dir   = r"C:\Users\Cheryl\OneDrive - National University of Singapore\NUS_CN\GE5219-SPATIAL PROGRAMMING\03_Projects\FINAL"

os.makedirs(output_dir, exist_ok=True) #ensure that output folder exists before saving files within
vector_proj   = os.path.join(gdb_path, "CDR_Buffers_WGS84")
vector_shp_dir = os.path.join(output_dir, "CDR_ComfortBuffers_WGS84_SHP")
vector_shp = os.path.join(vector_shp_dir, "CDR_ComfortBuffers_WGS84.shp")
zip_path = os.path.join(output_dir, "CDR_ComfortBuffers_WGS84.zip")

# Remove remaining files from previous runs for conflict management 
# This is to ensure that ArcPy would not throw erros given its sensitivity of naming and locks 

# Looping through the known file name outputs, seeing if the dataset exists & deleting it to avoid conflicts
for fc in [join_numeric, buffer_fc, sg_local, vector_proj]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

# Important realisation is that a shapefile needs a fresh set of matching file components
# Any leftover files would risk the shape export
# Checking if export folder exists & deleting all files inside.If not, a new file is created.
if os.path.exists(vector_shp_dir):
    for f in os.listdir(vector_shp_dir):
        os.remove(os.path.join(vector_shp_dir, f))
else:
    os.makedirs(vector_shp_dir, exist_ok=True)

# Remocing the old zipped shapedfile in order for mapbox uploading to be smooth 
if os.path.exists(zip_path):
    os.remove(zip_path)

#####  3C) SINGAPORE BOUNDARY MASKING


In [ ]:
print("Fetching SG boundary...") # message to indicate that this process started for tracking purpose 

arcpy.management.CopyFeatures(sg_boundary, sg_local) # downloads singapore boundary feature layer from ArcGIS Online to local GDB

# Check if projection of the layer matches SVY21
desc_sg = arcpy.Describe(sg_local) 
if not desc_sg.spatialReference or desc_sg.spatialReference.factoryCode != 3414: 
    sg_local_3414 = os.path.join(gdb_path, "SG_Boundary_Local_3414")
    #if projection does not match, reprojection is triggered accordingly 
    arcpy.management.Project(sg_local, sg_local_3414, arcpy.SpatialReference(3414))
    arcpy.management.Delete(sg_local)
    sg_local = sg_local_3414

# All Spatial Operations are bounded to Singapore Boundary through set extent and masking 
arcpy.env.extent = arcpy.Describe(sg_local).extent
arcpy.env.mask = sg_local

#####  3D) FETCH STATIONS & CROWD TABLE TO JOIN AND SCORE 

In [ ]:
print("Downloading station points & crowd table…") #statement indicating process has started

#defining temporary file paths 
temp_stations = os.path.join(gdb_path, "Stations_Temp") #downloaded MRT station geometry
temp_crowd    = os.path.join(gdb_path, "Crowd_Temp") #the downloaded crowd lookup table

# Deleting temporary files if exist to ensure clean workspace
for fc in [temp_stations, temp_crowd, join_numeric]:
    if arcpy.Exists(fc):
        arcpy.management.Delete(fc)

# Use of for loop to retrieve both hosted layers (station point geometry & crowd table ) into geodatabase
for i in range(2):
    try:
        arcpy.conversion.FeatureClassToFeatureClass(station_points, gdb_path, os.path.basename(temp_stations))
        arcpy.conversion.TableToTable(crowd_table, gdb_path, os.path.basename(temp_crowd))
        break

# Retry Mechanism setup automation stability, in case of a network fetch fail  
    except Exception as e:
        print(f"Retry fetch in 10s...({e})")
        time.sleep(10)

# To allow distance tools to run correctly, projection check to ensure all is in SVY21 (EPSG:3414) 
desc_st = arcpy.Describe(temp_stations)
if not desc_st.spatialReference or desc_st.spatialReference.factoryCode != 3414:
    # if projection is indeed wrong, stations are reprojected and a new updated feature class would be created
    # the old version would be deleted and replaced 
    st_3414 = os.path.join(gdb_path, "Stations_Temp_3414")
    arcpy.management.Project(temp_stations, st_3414, arcpy.SpatialReference(3414))
    arcpy.management.Delete(temp_stations)
    temp_stations = st_3414

# To ensure the accurate join key is found regardless of case or formatting method
def _find_field_case_insensitive(table, name):
    for f in arcpy.ListFields(table):
        if f.name.lower() == name.lower():
            return f.name
    return None

# To get accurate "Station" field in the tables for station geometry and crowd table
# Matching is important for the join to work 
st_key = _find_field_case_insensitive(temp_stations, "Station")
tb_key = _find_field_case_insensitive(temp_crowd, "Station")

# Station Points are copied as working file where joined crowd attributes are added
arcpy.management.CopyFeatures(temp_stations, join_numeric)

# Crowd Values are joined into station geometry
arcpy.management.JoinField(join_numeric, st_key, temp_crowd, tb_key)

# Find 'CrowdLevel' column regardless of of case or formatting method
cand = [f.name for f in arcpy.ListFields(join_numeric) if f.name.lower().endswith("crowdlevel")] \
       or [f.name for f in arcpy.ListFields(join_numeric) if f.name.lower() == "crowdlevel"]
crowd_field = cand[0]

# Add numeric field for our standardised (0-100) classification across the maps 
print("Converting L/M/H → 100/50/0…")
if "Crowd_Comfort" not in [f.name for f in arcpy.ListFields(join_numeric)]:
    arcpy.management.AddField(join_numeric, "Crowd_Comfort", "DOUBLE")

# Conversion logic (l,m,h >>> numeric- 100,50,0)
expr = f"score(!{crowd_field}!)"
code_block = r"""
def score(v):
    if v is None: return None
    s = str(v).strip().lower()
    if s in ('l','lo','low','1'):      return 100.0
    if s in ('m','mid','med','medium','2'): return 50.0
    if s in ('h','hi','high','3'):     return 0.0
    return None
"""

# Scoring logic is applied to dataset: Each station would have a associated Crowd_Comfort numeric value 
arcpy.management.CalculateField(join_numeric, "Crowd_Comfort", expr, "PYTHON3", code_block)

#####  3E) CREATION OF 300M (5-MIN ACCESSUBILITY) BUFFER IN SVY21

In [ ]:
print("Creating 300 m buffers around stations…") # message printed to show that the buffer creation process has started 

# To ensure no remaining buffer feature classed from previous runs for a clean write
if arcpy.Exists(buffer_fc):
    arcpy.management.Delete(buffer_fc)

# Creation of 300m Euclidean buffers around each station (station influence zones)
# Joined crowd comfort values are also integrated within the new buffer attributes
# Dissolve kept to none to ensure each station has its own individual polygon
arcpy.analysis.Buffer(join_numeric, buffer_fc, "300 Meters", dissolve_option="NONE")

# In preparation for Mapbox symbology representation using categotial text class for clearer visualisation 
print("Adding categorical comfort class…") # message to update status where script moves toward symbolisation preparation 

# Checks if the current output layer has pre-existing categorial text, if not it creates one 
if "Comfort_Class" not in [f.name for f in arcpy.ListFields(buffer_fc)]:
    arcpy.management.AddField(buffer_fc, "Comfort_Class", "TEXT", field_length=10)

# Numeric thresholds are used to inform textual comfort class ("High"/ "Medium"/"Low")
arcpy.management.CalculateField(
    buffer_fc, "Comfort_Class",
    "('Low' if !Crowd_Comfort! < 40 else ('Medium' if !Crowd_Comfort! <70 else 'High'))",
    "PYTHON3"
)

### STEP 04: UPLOAD ONTO MAPBOX FOR AUTOMATED MAP REFRESH

#####  4A) PROJECTION TO WGS84 AND EXPORT TO SHAPEFILE ZIP FILE

In [ ]:
print("Reprojecting buffers to WGS84 for Mapbox...") # message that signals the beginning of the projection 

# Ensures that there is no old versions, corruptions or naming conflicts from previous runs 
if arcpy.Exists(vector_proj):
    arcpy.management.Delete(vector_proj)
# Creation of new feature class that is reprojected to WGS84 for Mapbox environment
arcpy.management.Project(buffer_fc, vector_proj, arcpy.SpatialReference(4326)) # works as (input, output, coordinate system)

# Export as Shapefile format to be compatible with Mapbox environment
print("Exporting to Shapefile (.shp)…")
arcpy.conversion.FeatureClassToShapefile(
    Input_Features=[vector_proj],
    Output_Folder=vector_shp_dir
)

# Zip all Shapefile components
print("Zipping shapefile for upload…")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for f in os.listdir(vector_shp_dir):
        fp = os.path.join(vector_shp_dir, f)
        zipf.write(fp, arcname=f)

# Final Output Message 
# To verify that the pipeline as successfully ran to this stage 
# To signal that the export is finish for the uploading process to commence for verification 

print(f"""
COMPLETED SUCCESSFULLY
- 300 m buffers 
- Attributes: Crowd_Comfort, Comfort_Class
- WGS84 projection
- Auto-overwrites existing .shp and .zip
-Ready for Mapbox/QGIS upload
> {zip_path}
""")

#####  4B) UPLOADING TO MAPBOX

In [ ]:
print("Uploading to Mapbox...") #print message showing start of the progress 

# Defining Mapbox credentials & the necessary paths 
MAPBOX_USERNAME = "<username input-redacted due to privacy>"
MAPBOX_TOKEN = "sk.eyJ1IjoiZTE1NTM1MjIiLCJhIjoiY21obXkwNmJsMXRsZDJtcXZzcjIzaDVsMiJ9.c60OMrW8YSl_7Z7NFO4cyA"
TILESET_ID = "e1553522.1gf3nmjq"   
TILESET_NAME = "CDR_ComfortBuffers_WGS84"
zip_to_upload = os.path.join(output_dir, "CDR_ComfortBuffers_WGS84.zip")

# Calls Mapbox's "upload credentials" API 
# To request and obtain temporary AWS credentials that is needed to upload to temporary AWS S3 bucket 
url_request = f"https://api.mapbox.com/uploads/v1/{MAPBOX_USERNAME}/credentials?access_token={MAPBOX_TOKEN}"
resp = requests.post(url_request)
if resp.status_code != 200:
    raise RuntimeError(f" Failed to get upload credentials: {resp.text}")
creds = resp.json()

# Upload shapefile ZIP from my computer environment into Mapbox S3 
print(" Uploading shapefile ZIP to Mapbox ...")
try:
    s3_client = boto3.client(
        "s3",
        aws_access_key_id=creds["accessKeyId"],
        aws_secret_access_key=creds["secretAccessKey"],
        aws_session_token=creds["sessionToken"],
        region_name="us-east-1"
    )

    s3_client.upload_file(zip_to_upload, creds["bucket"], creds["key"])
    print(" File uploaded successfully.")
# Error handling: ensure that error messages would be sent if failed for debugging 
except NoCredentialsError:
    raise RuntimeError(" Missing credentials for upload.")
except Exception as e:
    raise RuntimeError(f" Upload failed: {str(e)}")

# Link to Mapbox to process this new upload as the defined tileset and update accordingly 
upload_payload = {
    "url": creds["url"],
    "tileset": TILESET_ID,
    "name": TILESET_NAME
}
upload_url = f"https://api.mapbox.com/uploads/v1/{MAPBOX_USERNAME}?access_token={MAPBOX_TOKEN}"
final_resp = requests.post(upload_url, json=upload_payload)

# Ensures that the upload request succeed, if not a message is sent for debugging
if final_resp.status_code not in [200, 201]:
    raise RuntimeError(f" Mapbox upload failed: {final_resp.text}")

# Retrieval of unique job ID to verify progress of the the upload job
upload_job = final_resp.json()
upload_id = upload_job.get("id", None)
print(f" Upload started successfully for tileset: {TILESET_ID}")

# Check Mapbox API until upload is process
if upload_id:
    print(" Waiting for Mapbox to finish processing...")
    status_url = f"https://api.mapbox.com/uploads/v1/{MAPBOX_USERNAME}/{upload_id}?access_token={MAPBOX_TOKEN}"

# A loop is created for up to 15 cycles, every 10 seconds 
# To query "upload status"
# To check if comversion is completed or there are errors
    for i in range(15):
        time.sleep(10)  
        status_resp = requests.get(status_url)
        status_json = status_resp.json()
# Stopping conditions defined below 
        if status_json.get("complete"):
            print(f" Mapbox processing complete! Tileset ready to view.")
            break
        elif status_json.get("error"):
            raise RuntimeError(f" Mapbox processing error: {status_json['error']}")
        else:
            print(f" Still processing ({i+1}/15)...")


# Print final message showing completion 
# Included indicators like the id of the specific mapbox tileset
# A link is also added for one to check if the overwrite went through
print(f"""
Upload completed successfully!
-Tileset ID: {TILESET_ID} .
-Auto-overwrite completed.
   https://studio.mapbox.com/tilesets/{TILESET_ID}
""")